In [1]:
import pandas as pd

In [2]:
# preprocessing.py

import os
import torch
import numpy as np
import logging
from torch.utils.data import Dataset

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

MAX_LEN = 512

USECOLS = [
    'sequence',
    'cdr3_start','cdr3_end',
    'v_sequence_start','v_sequence_end',
    'j_sequence_start','j_sequence_end',
    'v_score','v_identity',
    'j_score','j_identity',
    'locus','sequence_id', 'rev_comp'
]

CHAR_TO_IDX = {'A':0,'C':1,'G':2,'T':3,'N':4,'P':5}
IDX_TO_CHAR = ['A','C','G','T','N','P']

In [3]:
def load_and_aggregate(ig_path, tcr_path):

    logger.info("=" * 60)
    logger.info("STARTING IG/TCR AGGREGATION")
    logger.info("=" * 60)

    CHUNK = 1_000_000

    def process_file(path):

        logger.info(f"Processing file: {path}")

        dfs = []
        total_chunks = 0
        total_rows = 0

        for chunk_id, chunk in enumerate(
            pd.read_csv(
                path,
                usecols=USECOLS,
                delimiter="\t",
                chunksize=CHUNK
            )
        ):

            total_chunks += 1
            total_rows += len(chunk)

            logger.info(
                f"[{os.path.basename(path)}] "
                f"Chunk {chunk_id+1} | rows={len(chunk):,}"
            )

            v = (
                chunk.sort_values("v_identity", ascending=False)
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "v_sequence_start",
                    "v_sequence_end",
                    "v_score",
                    "v_identity"
                ]]
            )

            j = (
                chunk.sort_values("j_identity", ascending=False)
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "j_sequence_start",
                    "j_sequence_end",
                    "j_score",
                    "j_identity"
                ]]
            )

            cdr3 = (
                chunk.dropna(subset=["cdr3_start"])
                .drop_duplicates("sequence_id")
                [[
                    "sequence_id",
                    "cdr3_start",
                    "cdr3_end"
                ]]
            )

            seq = (
                chunk[["sequence_id","sequence"]]
                .drop_duplicates("sequence_id")
            )

            df = seq.merge(v, how="left", on="sequence_id")
            df = df.merge(j, how="left", on="sequence_id")
            df = df.merge(cdr3, how="left", on="sequence_id")

            dfs.append(df)

        logger.info(
            f"Finished {path} | "
            f"chunks={total_chunks} | "
            f"rows={total_rows:,}"
        )

        result = pd.concat(dfs)

        logger.info(
            f"Aggregated dataframe shape: {result.shape}"
        )

        return result

    ig = process_file(ig_path)
    tcr = process_file(tcr_path)

    logger.info("Computing quality scores...")

    def score(df):
        return (
            df.v_score.fillna(0) +
            df.j_score.fillna(0) +
            2*(df.v_identity.fillna(0) + df.j_identity.fillna(0))
        )

    ig["score"] = score(ig)
    tcr["score"] = score(tcr)

    logger.info("Merging IG/TCR datasets...")

    merged = pd.merge(
        ig,
        tcr,
        on="sequence_id",
        how="outer",
        suffixes=("_ig","_tcr")
    )

    logger.info(f"Merged shape: {merged.shape}")

    choose_ig = (
        merged.score_ig.fillna(-1)
        >=
        merged.score_tcr.fillna(-1)
    )

    final = pd.DataFrame({
        "sequence": np.where(
            choose_ig,
            merged.sequence_ig,
            merged.sequence_tcr
        ),

        "sequence_id": merged.sequence_id,

        "v_sequence_start": np.where(
            choose_ig,
            merged.v_sequence_start_ig,
            merged.v_sequence_start_tcr
        ),

        "v_sequence_end": np.where(
            choose_ig,
            merged.v_sequence_end_ig,
            merged.v_sequence_end_tcr
        ),

        "j_sequence_start": np.where(
            choose_ig,
            merged.j_sequence_start_ig,
            merged.j_sequence_start_tcr
        ),

        "j_sequence_end": np.where(
            choose_ig,
            merged.j_sequence_end_ig,
            merged.j_sequence_end_tcr
        ),

        "cdr3_start": np.where(
            choose_ig,
            merged.cdr3_start_ig,
            merged.cdr3_start_tcr
        ),

        "cdr3_end": np.where(
            choose_ig,
            merged.cdr3_end_ig,
            merged.cdr3_end_tcr
        ),

        "best_v_identity": np.where(
            choose_ig,
            merged.v_identity_ig,
            merged.v_identity_tcr
        ),

        "best_j_identity": np.where(
            choose_ig,
            merged.j_identity_ig,
            merged.j_identity_tcr
        ),

        "best_v_score": np.where(
            choose_ig,
            merged.v_score_ig,
            merged.v_score_tcr
        ),

        "best_j_score": np.where(
            choose_ig,
            merged.j_score_ig,
            merged.j_score_tcr
        ),
    })

    final["has_v"] = final.v_sequence_start.notna()
    final["has_j"] = final.j_sequence_start.notna()

    logger.info("=" * 60)
    logger.info("FINAL DATASET SUMMARY")
    logger.info("=" * 60)
    logger.info(f"Final shape: {final.shape}")
    logger.info(f"Has V: {final.has_v.sum():,}")
    logger.info(f"Has J: {final.has_j.sum():,}")

    return final

In [9]:
data_10245671 = load_and_aggregate('result_10245671.tsv','result_10245671_tcr.tsv')
data_10245671.to_csv('test_10245671_reference.tsv')

INFO:__main__:============================================================
INFO:__main__:STARTING IG/TCR AGGREGATION
INFO:__main__:============================================================
INFO:__main__:Processing file: result_10245671.tsv
INFO:__main__:[result_10245671.tsv] Chunk 1 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 2 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 3 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 4 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 5 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 6 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 7 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 8 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 9 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 10 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 11 | rows=1,000,000
INFO:__main__:[result_10245671.tsv] Chunk 12 | rows=1,000,000
INFO:__main__:[result_

In [1]:
data_18748718 = load_and_aggregate('train_18748718.tsv','train_18748718_tcr.tsv')
data_18748718.to_csv('test_18748718_reference.tsv')

NameError: name 'load_and_aggregate' is not defined

In [12]:
data_18748718 = load_and_aggregate('train_5494024.tsv','train_5494024_tcr.tsv')
data_18748718.to_csv('test_5494024_reference.tsv')

INFO:__main__:============================================================
INFO:__main__:STARTING IG/TCR AGGREGATION
INFO:__main__:============================================================
INFO:__main__:Processing file: train_5494024.tsv
INFO:__main__:[train_5494024.tsv] Chunk 1 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 2 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 3 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 4 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 5 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 6 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 7 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 8 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 9 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 10 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 11 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 12 | rows=1,000,000
INFO:__main__:[train_5494024.tsv] Chunk 13 | row

In [13]:
data_18748718 = load_and_aggregate('result_18748717.tsv','result_18748717_tcr.tsv')
data_18748718.to_csv('test_18748717_reference.tsv')

INFO:__main__:============================================================
INFO:__main__:STARTING IG/TCR AGGREGATION
INFO:__main__:============================================================
INFO:__main__:Processing file: result_18748717.tsv
INFO:__main__:[result_18748717.tsv] Chunk 1 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 2 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 3 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 4 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 5 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 6 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 7 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 8 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 9 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 10 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 11 | rows=1,000,000
INFO:__main__:[result_18748717.tsv] Chunk 12 | rows=1,000,000
INFO:__main__:[result_

In [ ]:
5494024 4482.12 18748717 9165.57 18748718 11343.38 10245671 4044.50

In [ ]:
5494024 1492.08 18748717 1959.07 18748718 2015.35 10245671 938.39

In [3]:
df = pd.read_csv('result_15301160.tsv', delimiter='\t')

In [18]:
df.iloc[0]['sequence']

'CCCATCACCAAAATGCTGGGGCTGATTGCTATCGCCTGTCCCGGTGGCACAGAAGTACACAGATGTCTGGGAGCTGGT'

In [21]:
df.iloc[0]['sequence_alignment']

'CTGGGGCTGATTGCTATCGCCTGTCCCGGTGGCACAGAAGTACACAGATGTCTGGGAGCTG'

In [22]:
df.iloc[0]['sequence_aa']

'ADCYRLSRWHRSTQMSGSW'

In [ ]:
df.iloc[0]['v_sequence_start']

In [ ]:
df.iloc[0]['v_sequence_end']

In [17]:
df.iloc[0].index

Index(['sequence_id', 'sequence', 'sequence_aa', 'locus', 'stop_codon',
       'vj_in_frame', 'v_frameshift', 'productive', 'rev_comp', 'complete_vdj',
       'd_frame', 'v_call', 'd_call', 'j_call', 'sequence_alignment',
       'germline_alignment', 'sequence_alignment_aa', 'germline_alignment_aa',
       'v_alignment_start', 'v_alignment_end', 'd_alignment_start',
       'd_alignment_end', 'j_alignment_start', 'j_alignment_end',
       'v_sequence_alignment', 'v_sequence_alignment_aa',
       'v_germline_alignment', 'v_germline_alignment_aa',
       'd_sequence_alignment', 'd_sequence_alignment_aa',
       'd_germline_alignment', 'd_germline_alignment_aa',
       'j_sequence_alignment', 'j_sequence_alignment_aa',
       'j_germline_alignment', 'j_germline_alignment_aa', 'fwr1', 'fwr1_aa',
       'cdr1', 'cdr1_aa', 'fwr2', 'fwr2_aa', 'cdr2', 'cdr2_aa', 'fwr3',
       'fwr3_aa', 'fwr4', 'fwr4_aa', 'cdr3', 'cdr3_aa', 'junction',
       'junction_length', 'junction_aa', 'junction_aa_leng